In [1]:
import os
from tqdm import tqdm

MODE = 'test'
VVENC = '/home/zhaoy/vvenc/bin/release-static/vvencapp'
QPs = list(range(22, 52, 5))
PRESET = 'medium'
DEBUG  = False

In [2]:
import time

def enc_vvenc(size, yuv_dir, rlt_root, vvenc=VVENC, qps=QPs, preset=PRESET, debug=DEBUG):
    yuvs = [_ for _ in os.listdir(yuv_dir) if _.endswith('.yuv')]
    for yuv in tqdm(yuvs):
        yuv_path = os.path.join(yuv_dir, yuv)
        os.makedirs(f'{rlt_root}/log', exist_ok=True)
        os.makedirs(f'{rlt_root}/bin', exist_ok=True)
        
        for qp in qps:
            log_path = os.path.join(rlt_root, 'log', yuv.replace('.yuv', f'_qp{qp}_{PRESET}.log'))
            bin_path = os.path.join(rlt_root, 'bin', yuv.replace('.yuv', f'_qp{qp}_{PRESET}.bin'))
            
            cmd = f'{vvenc} --size {size} --preset {preset} --qp {qp} --fps 30 --format yuv420 --input {yuv_path} --passes 1 --output {bin_path} --threads 8 > {log_path} &'
            os.system(cmd)
            if DEBUG:
                break
        time.sleep(3)

In [ ]:
''' 1. Touch and Go '''
yuv_root = '/data/ssd/zhaoy/datasets/TouchandGoDataset-v2/dataset-comp'
yuv_dir  = os.path.join(yuv_root, MODE, 'video', 'yuv')
rlt_root = '/data/ssd/zhaoy/datasets/TouchandGoDataset-v2/compressed/vvenc'

# enc_vvenc(size='640x480', yuv_dir=yuv_dir, rlt_root=rlt_root)

In [ ]:
''' 2. Object Folder '''
yuv_root = '/data/ssd/zhaoy/datasets/ObjectFolder_1.0/dataset-comp'
yuv_dir  = os.path.join(yuv_root, MODE, 'video', 'yuv')
rlt_root = '/data/ssd/zhaoy/datasets/ObjectFolder_1.0/compressed/vvenc'

# enc_vvenc(size='160x120', yuv_dir=yuv_dir, rlt_root=rlt_root)

  0%|          | 0/200 [00:00<?, ?it/s]

 15%|█▌        | 30/200 [01:04<06:07,  2.16s/it]

In [ ]:
''' 3. SSVTP '''
yuv_root = '/data/ssd/zhaoy/datasets/SSVTP/dataset-comp'
yuv_dir  = os.path.join(yuv_root, MODE, 'video')
rlt_root = '/data/ssd/zhaoy/datasets/SSVTP/compressed/vvenc'

# enc_vvenc(size='240x320', yuv_dir=yuv_dir, rlt_root=rlt_root)

100%|██████████| 1/1 [00:10<00:00, 10.03s/it]


In [6]:
''' 4. YCB-Slide '''
yuv_root = '/data/ssd/zhaoy/datasets/YCB-Slide/dataset-comp'
yuv_dir  = os.path.join(yuv_root, MODE, 'video', 'yuv')
rlt_root = '/data/ssd/zhaoy/datasets/YCB-Slide/compressed/vvenc'

enc_vvenc(size='240x320', yuv_dir=yuv_dir, rlt_root=rlt_root)

100%|██████████| 4/4 [00:12<00:00,  3.01s/it]


In [ ]:
import re

def extract_vvenc_metrics(log_path):
    with open(log_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    bitrate, y_psnr, u_psnr, v_psnr, yuv_psnr = None, None, None, None, None
    for line in reversed(lines):
        if line.startswith('vvenc [info]:\tTotal Frames'):
            idx = lines.index(line)
            if idx + 1 < len(lines):
                vals = re.split(r'\s+', lines[idx+1].strip())
                bitrate = float(vals[4])
                y_psnr = float(vals[5])
                u_psnr = float(vals[6])
                v_psnr = float(vals[7])
                yuv_psnr = (6*y_psnr + u_psnr + v_psnr)/8
    elapsed_time = None
    for line in lines:
        if line.startswith('vvencapp [info]: Total Time:'):
            m = re.search(r'Total Time:\s*([\d\.]+)', line)
            if m:
                elapsed_time = float(m.group(1))
    return [elapsed_time, bitrate, yuv_psnr]